# Agent Evaluation & LLM-as-a-Judge
Evaluating non-deterministic agents is notoriously difficult. Standard unit tests (`assert response == "Yes"`) fail because LLMs phrase answers differently every time.

This notebook demonstrates SOTA evaluation techniques:
1. **LLM-as-a-Judge (Outcome Scoring):** Using a superior LLM (like GPT-4) to grade the output of a cheaper agent using strict rubrics.
2. **Trajectory Evaluation:** Scoring the *steps* the agent took, not just the final answer.

**Dependencies required:** `pip install pydantic`


## 1. LLM-as-a-Judge (Outcome Scoring)
We use `pydantic` to force the "Judge" LLM to output a strict binary score and a justification.


In [1]:
from pydantic import BaseModel, Field
# 1. Define the Judge's Schema
import sys, os; sys.path.insert(0, os.path.join(os.getcwd(), 'curriculum/intermediate/05-agent-evaluation')); from policy import EvaluationScore
# 2. Define the Evaluation Rubric
eval_rubric = """
You are an impartial judge evaluating an AI customer service agent.
The agent must:
1. Apologize if the user is frustrated.
2. Provide the exact order status.
"""
# 3. Simulate the Judge's Assessment
def run_llm_judge(user_input: str, agent_response: str, actual_status: str) -> EvaluationScore:
    print(f"⚖️ [Judge] Evaluating Response...")
    print(f"   User: '{user_input}'")
    print(f"   Agent: '{agent_response}'")
    print(f"   Ground Truth Status: '{actual_status}'")
    
    # In reality, you pass the rubric, user_input, agent_response, and ground truth to GPT-4o.
    # We mock the LLM parsing logic here:
    if "apologize" not in agent_response.lower() and "sorry" not in agent_response.lower():
        return EvaluationScore(is_correct=False, justification="The agent failed to apologize to the frustrated user.")
    
    if actual_status.lower() not in agent_response.lower():
        return EvaluationScore(is_correct=False, justification="The agent provided the wrong order status.")
        
    return EvaluationScore(is_correct=True, justification="The agent apologized and provided the correct status.")
# --- Test Case 1: Agent Fails ---
score1 = run_llm_judge(
    user_input="Where is my package?! I've been waiting for weeks!",
    agent_response="It is shipped.",
    actual_status="Shipped"
)
print(f"\n📊 Result: {'✅ Pass' if score1.is_correct else '❌ Fail'}")
print(f"📝 Justification: {score1.justification}\n")
# --- Test Case 2: Agent Succeeds ---
score2 = run_llm_judge(
    user_input="Where is my package?! I've been waiting for weeks!",
    agent_response="I am so sorry for the delay! I checked the database and your order is currently Shipped.",
    actual_status="Shipped"
)
print(f"\n📊 Result: {'✅ Pass' if score2.is_correct else '❌ Fail'}")
print(f"📝 Justification: {score2.justification}")


⚖️ [Judge] Evaluating Response...
   User: 'Where is my package?! I've been waiting for weeks!'
   Agent: 'It is shipped.'
   Ground Truth Status: 'Shipped'

📊 Result: ❌ Fail
📝 Justification: The agent failed to apologize to the frustrated user.

⚖️ [Judge] Evaluating Response...
   User: 'Where is my package?! I've been waiting for weeks!'
   Agent: 'I am so sorry for the delay! I checked the database and your order is currently Shipped.'
   Ground Truth Status: 'Shipped'

📊 Result: ✅ Pass
📝 Justification: The agent apologized and provided the correct status.


## 2. Trajectory Evaluation (Scoring the Path)
Sometimes the final answer is correct, but the agent took a horrible, inefficient path to get there (e.g., hallucinating tools, looping 5 times). 

Trajectory Evaluation requires you to log the `AgentAction` trace and pass the *entire array* to the Judge LLM.


In [2]:
import sys, os; sys.path.insert(0, os.path.join(os.getcwd(), 'curriculum/intermediate/05-agent-evaluation')); from policy import TrajectoryScore
# 1. Simulate an Agent's Execution Trace
inefficient_trace = [
    {"thought": "I need to check order 123.", "action": "sql_query(SELECT * FROM orders)"},
    {"observation": "Error: Unbounded query. Too many results."},
    {"thought": "I will try again.", "action": "sql_query(SELECT * FROM orders WHERE id=123)"},
    {"observation": "Status: Delivered"},
    {"thought": "I have the answer.", "action": "final_answer('Your order is delivered.')"}
]
efficient_trace = [
    {"thought": "I need to check order 123. I must restrict the query.", "action": "get_order_status(id=123)"},
    {"observation": "Status: Delivered"},
    {"thought": "I have the answer.", "action": "final_answer('Your order is delivered.')"}
]
def evaluate_trajectory(trace: list[dict]) -> TrajectoryScore:
    # In reality, pass the trace JSON to the Judge LLM.
    # We mock the Judge spotting errors:
    for step in trace:
        if "Error:" in step.get("observation", ""):
            return TrajectoryScore(is_efficient=False, penalty_reason="Agent triggered a database error due to an unbounded query.")
    return TrajectoryScore(is_efficient=True)
print("--- Evaluating Inefficient Agent ---")
score = evaluate_trajectory(inefficient_trace)
print(f"📊 Efficient: {score.is_efficient} | Penalty: {score.penalty_reason}")
print("\n--- Evaluating Efficient Agent ---")
score = evaluate_trajectory(efficient_trace)
print(f"📊 Efficient: {score.is_efficient} | Penalty: {score.penalty_reason}")


--- Evaluating Inefficient Agent ---
📊 Efficient: False | Penalty: Agent triggered a database error due to an unbounded query.

--- Evaluating Efficient Agent ---
📊 Efficient: True | Penalty: None
